In [1]:
import pandas as pd;

# Get All CSV data from Scraped Marketplaces Products

In [2]:
import glob 
from pathlib import Path

folder_path = "datasets/marketplace_products"
marketplace_folders = Path(folder_path)

for dir in marketplace_folders.iterdir():
    print(dir)

datasets\marketplace_products\shopee
datasets\marketplace_products\tokopedia


In [3]:
phone_products_ls = []

for folder in marketplace_folders.iterdir():
    for csv_file in folder.glob("*.csv"):
        phone_products_df = pd.read_csv(csv_file)
        phone_products_ls.append(phone_products_df)

In [4]:
print(f"found {len(phone_products_ls)} CSV files")
phone_products_df = pd.concat(phone_products_ls, ignore_index=True)
print(len(phone_products_df))
phone_products_df.sample()

found 48 CSV files
32125


,brand,title,price,location,rating,sold
19875,Oppo,"Oppo A5s (AX5s) 3GB/32GB - Standard, No Box",Rp600.000,NaN,5.0,10 terjual


# Get All Phones Spesifications Data

In [5]:
file_path = "datasets/phone_specifications/phones_data.csv"

phone_specifications_ls = []
phone_specifications_df = pd.read_csv(file_path)

print(len(phone_specifications_df))

phone_specifications_df['model'] = phone_specifications_df['brand'] + " "+phone_specifications_df['model']
phone_specifications_df

4144


,brand,model,device_type,release_date,status,architecture,aspect_ratio,audio_jack,battery_capacity,battery_placement,...,usb_otg,usb_type_c,user_interface,video,volte,waterproof,weight,width,wifi_hotspot,wlan
0,Oppo,Oppo K13 Turbo Pro (512GB),Smartphone,25 July 2025,Available,64 bit,19.5:9,USB Type-C,Li-Po 7000mAh,NaN,...,Yes,USB Type-C 2.0,ColorOS 15,"4K@30/60fps, 1080p@30fps",Yes,Water resistant (up to 2m for 30 min),208 grams,77.2 mm,NaN,"Wi-Fi 7 (802.11 a/b/g/n/ac/be/ax) 5GHz 6GHz, MIMO"
1,Oppo,Oppo K13 Turbo (512GB),Smartphone,25 July 2025,Available,64 bit,19.5:9,USB Type-C,Li-Po 7000mAh,NaN,...,Yes,USB Type-C 2.0,ColorOS 15,"4K@30/60fps, 1080p@30fps",Yes,Water resistant (up to 2m for 30 min),207 grams,77.2 mm,NaN,"Wi-Fi 6 (802.11 a/b/g/n/ac/ax) 5GHz, MIMO"
2,Vivo,Vivo Y400,Smartphone,Not announced yet,Rumored,64 bit,NaN,USB Type-C,Li-Po 6000mAh,NaN,...,Yes,USB Type-C 2.0,Funtouch 15,"1080p@30fps, gyro-EIS",Yes,Water resistant (up to 1.5m for 30 min),NaN,NaN,NaN,Wi-Fi 5 (802.11 a/b/g/n/ac) 5GHz
3,Infinix,Infinix Hot 60 Pro Plus (256GB),Smartphone,25 July 2025,Available,64 bit,20:9,3.5 mm,Li-Po 5160mAh,NaN,...,Yes,USB Type-C 2.0,XOS 15.1,"1440p@30fps, 1080p@30/60fps",Yes,Splash proof,155 grams,75.8 mm,NaN,Wi-Fi 5 (802.11 a/b/g/n/ac) 5GHz
4,Realme,Realme Note 70T,Smartphone,Exp. 31 July 2025,Upcoming,64 bit,20:9,3.5 mm,Li-Po 6000mAh,NaN,...,Yes,USB Type-C 2.0,Realme UI 5.0,1080p@30fps,Yes,Splash proof,201 grams,76.6 mm,NaN,Wi-Fi 5 (802.11 a/b/g/n/ac) 5GHz
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4139,Xiaomi,Xiaomi Xiaomi Redmi A3,Smartphone,23 Feb 2024,Available,64 bit,20:9,3.5 mm,Li-Po 5000mAh,NaN,...,Yes,USB Type-C 2.0,MIUI,1080p@30fps,Yes,NaN,193 grams,76.3 mm,NaN,Wi-Fi 5 (802.11 a/b/g/n/ac) 5GHz
4140,Apple,Apple iPhone 15 Pro Max,Smartphone,22 September 2023,Available,64 bit,19.5:9,USB Type-C,Li-Ion 4441mAh,NaN,...,Yes,"USB Type-C 3.0, DisplayPort",NaN,"4K@24/25/30/60fps, 1080p@25/30/60/120/240fps, ...",Yes,Water resistant (up to 30 minutes in a depth o...,221 grams,76.7 mm,NaN,"Wi-Fi 6E (802.11 a/b/e/g/n/ac/ax) 5GHz 6GHz, MIMO"
4141,Samsung,Samsung Galaxy S23 Ultra,Smartphone,17 January 2023,Available,64 bit,19.3:9,USB Type-C,Li-Ion 5000mAh,NaN,...,Yes,USB Type-C 3.2,One UI 6.1.1,"8K@24/30fps, 4K@30/60fps, 1080p@30/60/240fps, ...",Yes,Water resistant (up to 30 minutes in a depth o...,233 grams,78.1 mm,NaN,"Wi-Fi 6 (802.11 a/b/g/n/ac/ax) 5GHz, MIMO"
4142,Comparisons,Comparisons Comparisons,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Preparing data to integrate

In [6]:
phone_brands = phone_products_df['brand'].unique()
phone_brands

<StringArray>
['Asus', 'Infinix', 'Oppo', 'Realme', 'Samsung', 'Tecno', 'Vivo', 'Xiaomi']
Length: 8, dtype: str

# Integrating All the Data Products with Phone Spesifications from Kaggle

In [7]:
from rapidfuzz import process, fuzz

In [8]:
def extract_candidate(title, brand, max_words):
    if pd.isna(title):
        return None
    
    title_lower = title.lower()
    brand_lower = brand.lower()
    idx = title_lower.find(brand_lower)
    
    if idx == -1:
        return None
    
    words = title[idx:].split()
    return " ".join(words[:max_words])

import pandas as pd
import re

def remove_ram_storage(text):
    if not isinstance(text, str):
        return text
    
    text = text.lower()
    text = re.sub(r'\b\d+\s*(?:gb|tb|mb)?\s*/\s*\d+\s*(?:gb|tb|mb)?\b', ' ', text)
    text = re.sub(r'\b\d+\s*(?:gb|tb|mb)\s*\+?\s*\d+\s*(?:gb|tb|mb)\b', ' ', text)
    text = re.sub(r'\b\d+\s*(?:gb|tb|mb)\b', ' ', text)
    
    return text.lower()

In [9]:
from os import remove


results = []

for brand in phone_brands:
    phone_products_to_integrate = phone_products_df[phone_products_df["brand"] == brand]
    phone_specifications_to_integrate = phone_specifications_df[phone_specifications_df["brand"] == brand]
    phone_specifications_to_integrate["model"] = (phone_specifications_to_integrate["model"].apply(lambda x: (x.lower())))
    
    get_max_length_model = phone_specifications_to_integrate["model"].str.split().str.len().max()
    
    if pd.isna(get_max_length_model):
        continue
    
    phone_products_to_integrate["candidate_model"] = (
        phone_products_to_integrate["title"].apply(lambda x: extract_candidate(x, brand, get_max_length_model))
    )
    
    phone_products_to_integrate["candidate_model"] = phone_products_to_integrate['candidate_model'].apply(remove_ram_storage)
    
    choices = phone_specifications_to_integrate["model"].tolist()
    # print(choices[:10])
    
    for _, row in phone_products_to_integrate.iterrows():
        match = process.extractOne(
            row['candidate_model'],
            choices,
            scorer=fuzz.token_set_ratio,
            score_cutoff=100
        )
        
        if match is None:
            continue
        
        matched_model, score, idx = match
        
        spec_row = phone_specifications_to_integrate.iloc[idx]
        
        merged = {
            **row.to_dict(),
            **spec_row.to_dict(),
            # "match_score" : score
        }
        
        results.append(merged)
        
merged_df = pd.DataFrame(results)

In [ ]:
# len(merged_df[merged_df['match_score'] > 80][['title','brand_w_ram_storage','match_score']])
len(merged_df)
merged_df['model']
# merged_df[merged_df['match_score'] == 100][['candidate_model','model','match_score']]


18424

# Just save data that get 100% score matching

In [11]:
merged_df.to_csv("datasets/integrated_phones/phones.csv", index=False)